# Spoken number alerts: a cyclone bulletin where the numbers survive

A cyclone warning is a set of numbers wearing a sentence. Wind at 110-120 km/h. Rainfall of
204.5 mm in 24 hours. Landfall between 06:00 and 09:00 on 30/08/2026. Dial 1077. The warning
arrives in English and has to go out in Odia, Telugu or Manipuri, and translation is where
numbers go wrong. A model that renders 1077 as 107 has produced fluent, confident text that
will make somebody dial a number which does not answer.

This notebook does four things:

1. Reads an English bulletin and pulls out every number in it.
2. Translates it twice, through mayura:v1 and through sarvam-translate:v1, and checks both
   translations against those numbers.
3. Renders the spoken form of the numbers, and shows why that rendering cannot be checked.
4. Speaks the alert where Sarvam has a voice, and prints a labelled card where it does not.

The checking runs offline in `alert_numbers.py`. Only the translating and the speaking need a key.

## Two things to know before you read the outputs

**This notebook has not been run against the live API.** There was no Sarvam API key on the
machine where it was written, so every code cell output below is empty. The calls were written
against the parameter names and docstrings of the installed sarvamai package. Run it yourself
with a key before trusting any of it.

**The bulletin in this recipe was authored for it and is not a real bulletin.** Nothing in it was
copied, adapted or paraphrased from an India Meteorological Department bulletin, a State Disaster
Management Authority release, or any other published warning; those are copyrighted. It names no
place, so it cannot be mistaken for a record of a real event. The candidate translations in
`AUDIT_FIXTURES` were authored here too, and none of them came from a live API call.

The design behind this recipe is written up in `docs/specs/spoken-number-alerts.md`.

In [ ]:
# Install the two packages this recipe needs. In a notebook cell, run:
#     %pip install -r requirements.txt
# From a terminal, run:
#     pip install -r requirements.txt

## Setup

The key is passed to the client explicitly. `SarvamAI.__init__` reads `SARVAM_API_KEY` in a
default argument, which Python evaluates once when the package is imported, so a `load_dotenv()`
that happens afterwards is too late and the client raises. Passing the key by hand is the only
reliable way.

In [ ]:
from __future__ import annotations

import base64
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI

sys.path.insert(0, str(Path.cwd()))
import alert_numbers as an

load_dotenv()

if not os.environ.get("SARVAM_API_KEY"):
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or in a .env file before running this notebook."
    )

client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Client ready.")

## 1. The bulletin

Put your own English bulletin in `sample_data/bulletin.txt` and this cell uses it instead of the
authored one. Nothing in `sample_data/` is committed.

In [ ]:
reader_supplied = Path("sample_data") / "bulletin.txt"

if reader_supplied.exists():
    bulletin = reader_supplied.read_text(encoding="utf-8")
    print("Using your bulletin from", reader_supplied)
else:
    bulletin = an.SOURCE_BULLETIN
    print("Using the authored bulletin that ships with this recipe.")

print(len(bulletin), "characters")
print(bulletin)

## 2. Every number in the source

Three kinds of number, because they fail in three different ways.

A **measurement** is a quantity: 204.5 mm, 42 camps, 12,000 people. It survives if its value
survives, so 45, 045 and a Devanagari rendering of forty-five are all the same number.

An **identifier** is something you dial: 1077, 108. It survives only if the digit string survives
exactly. A helpline that gains or loses a leading zero is a number that does not answer.

A **sequence** is ordered parts: 28/08/2026, 14:30, 110-120. It survives only if its parts come
through in the same order. 08/28/2026 holds every part of 28/08/2026 and means a different day.

Nothing in this section needs a key.

In [ ]:
facts = an.extract_number_facts(bulletin)
print(len(facts), "numbers found")

for fact in facts:
    unit = fact.unit if fact.unit else "-"
    print(f"  {fact.raw:<12} {fact.kind:<12} unit {unit:<8} at {fact.start}")

### The checker, on translations we wrote ourselves

These seven candidate translations are authored, not returned by any API call. Six of them are
Hindi renderings of the same bulletin: two correct, four broken in one place each. The seventh
writes every number out in words.

Read the summaries. They are what a district officer would read.

In [ ]:
for name, candidate in an.AUDIT_FIXTURES.items():
    report = an.audit_translation(facts, candidate)
    print(f"--- {name}: passed = {report.ok}")
    print(report.summary())
    print()

## 3. Which languages can be spoken, and which can only be printed

Sarvam translate reaches 23 language codes. Sarvam text to speech reaches 11. The twelve in
between can be translated but have no voice at all, so they get a printed card that says so.
They never get another language's voice.

Both rosters are read from the SDK's own type definitions when this runs, so a language that
gains a voice moves tier by itself.

Odia is `od-IN`. The rules file in this repo also lists `or-IN`, but the SDK's language type has
never contained it and the API rejects it, so the router refuses it and names `od-IN` instead.

In [ ]:
print("languages with a voice:", len(an.tts_language_codes()))
print("languages translate can write:", len(an.translate_language_codes()))
print("languages that translate but cannot speak:", len(an.text_card_language_codes()))

LANGUAGES = ["hi-IN", "od-IN", "ta-IN", "mni-IN", "ur-IN", "sat-IN"]
plans = an.plan_languages(LANGUAGES)

for plan in plans:
    print(
        f"  {plan.code:<8} {plan.delivery:<10} {plan.translate_model:<20}"
        f" cap {plan.char_cap} voice {plan.tts_voice}"
    )

try:
    an.plan_languages(["xx-IN"])
except an.UnsupportedLanguageError as error:
    print("refused:", error)

## 4. Segmenting, because the translate caps are the binding ones

mayura:v1 takes 1000 characters per call and sarvam-translate:v1 takes 2000. Both are smaller
than either text-to-speech cap, so the translate model a language routes to decides how the
bulletin is cut up.

The authored bulletin is 1152 characters, which is over one cap and under the other: the same
text needs two calls through one model and one call through the other. A cut is never allowed to
fall inside a number, because half a date in one call and half in the next is a date that has
been lost.

In [ ]:
for plan in plans:
    segments = an.segment_bulletin(bulletin, plan.char_cap)
    print(
        f"  {plan.code:<8} {plan.translate_model:<20} {len(segments)} segment(s)"
        f" of {[len(segment) for segment in segments]} characters"
    )

## 5. The three-way comparison

The same bulletin, three ways: through mayura:v1, through sarvam-translate:v1, and then through
the spoken-form rendering that a listener actually wants to hear. The first two are audited. The
third cannot be, and the reason is worth understanding.

### Arm 1: mayura:v1

In [ ]:
HINDI = "hi-IN"
hindi_plan = an.plan_languages([HINDI])[0]

mayura_parts = []
for segment in an.segment_bulletin(bulletin, an.MAYURA_CHAR_CAP):
    response = client.text.translate(
        input=segment,
        source_language_code="en-IN",
        target_language_code=HINDI,
        model="mayura:v1",
        mode="formal",
        numerals_format="international",
    )
    mayura_parts.append(response.translated_text)

mayura_text = "".join(mayura_parts)
print(mayura_text)

mayura_report = an.audit_translation(facts, mayura_text)
print()
print(mayura_report.summary())

### Arm 2: sarvam-translate:v1

One call instead of two, because this model takes 2000 characters. It supports formal mode only.

In [ ]:
sarvam_parts = []
for segment in an.segment_bulletin(bulletin, an.SARVAM_TRANSLATE_CHAR_CAP):
    response = client.text.translate(
        input=segment,
        source_language_code="en-IN",
        target_language_code=HINDI,
        model="sarvam-translate:v1",
        numerals_format="international",
    )
    sarvam_parts.append(response.translated_text)

sarvam_text = "".join(sarvam_parts)
print(sarvam_text)

sarvam_report = an.audit_translation(facts, sarvam_text)
print()
print(sarvam_report.summary())

### Arm 3: the spoken form, which cannot be checked

`transliterate(spoken_form=True)` turns numerals into the words a person would say. The SDK's own
example turns `9:30am` into a phrase with no digit left in it. That is exactly what you want a
loudspeaker to say, and it is also why the check has to happen on the arm above, before this one.
There is no digit left here to compare against the source.

`spoken_form_numerals_language` decides whether the numbers are said in English or in the target
language, so both are shown. Neither can be audited, and the auditor says so rather than
returning a green tick: it reports that a person must read the text and confirm the numbers by
hand. Note also that spoken form has no effect when the output language is en-IN.

In [ ]:
spoken_native = client.text.transliterate(
    input=mayura_text,
    source_language_code=HINDI,
    target_language_code=HINDI,
    spoken_form=True,
    spoken_form_numerals_language="native",
)
print("numbers spoken in the target language:")
print(spoken_native.transliterated_text)

spoken_english = client.text.transliterate(
    input=mayura_text,
    source_language_code=HINDI,
    target_language_code=HINDI,
    spoken_form=True,
    spoken_form_numerals_language="english",
)
print()
print("numbers spoken in English:")
print(spoken_english.transliterated_text)

spoken_report = an.audit_translation(facts, spoken_native.transliterated_text)
print()
print(spoken_report.summary())

## 6. Speaking it

Two arms, because two systems want two different files.

The **public-address arm** streams mulaw at 8000 Hz. 8 kHz mu-law is the telephony and
public-address convention; the SDK docstring constrains sample rates only for the OPUS codec and
says nothing about mulaw, so this pairing is **not confirmed** against the live API. If your PA
chain rejects it, that is the first thing to change.

The **IVR arm** asks for linear16 in one response instead of a stream.

Two things every call here does on purpose. It passes `model="bulbul:v3"`, because leaving the
model out sends nothing and the server then picks its own older default. And it passes
`language_code`, not `target_language_code`, which is the text-to-speech parameter name.

The streamed response is an iterator of bytes, so it is written out chunk by chunk as it arrives
rather than collected in memory first.

In [ ]:
if len(mayura_text) > an.TTS_STREAM_CHAR_CAP:
    raise RuntimeError(
        f"{len(mayura_text)} characters is over the streaming cap of {an.TTS_STREAM_CHAR_CAP};"
        " translate and speak one segment at a time."
    )

print("public-address arm:", an.PA_SYSTEM_CODEC, "at", an.PA_SYSTEM_SAMPLE_RATE, "Hz")

pa_path = OUTPUT_DIR / f"alert-{hindi_plan.code}-pa-mulaw-8000.raw"
written = 0

with pa_path.open("wb") as handle:
    for chunk in client.text_to_speech.convert_stream(
        text=mayura_text,
        language_code=hindi_plan.code,
        speaker=hindi_plan.tts_voice,
        model="bulbul:v3",
        output_audio_codec="mulaw",
        speech_sample_rate=8000,
    ):
        handle.write(chunk)
        written += len(chunk)

print("wrote", written, "bytes to", pa_path)

In [ ]:
if len(mayura_text) > an.TTS_CONVERT_CHAR_CAP:
    raise RuntimeError(
        f"{len(mayura_text)} characters is over the cap of {an.TTS_CONVERT_CHAR_CAP}"
        " for this endpoint; speak one segment at a time."
    )

print("IVR arm:", an.IVR_CODEC)

ivr = client.text_to_speech.convert(
    text=mayura_text,
    language_code=hindi_plan.code,
    speaker=hindi_plan.tts_voice,
    model="bulbul:v3",
    output_audio_codec="linear16",
)

ivr_path = OUTPUT_DIR / f"alert-{hindi_plan.code}-ivr-linear16.raw"
ivr_path.write_bytes(b"".join(base64.b64decode(part) for part in ivr.audios))
print("wrote", ivr_path)

## 7. Printing it, for the languages with no voice

For these languages there is no audio and there is no substitute. The card says so in words at
the top, carries the translated text, and carries the result of the number check. A card is never
rendered clean over a broken number.

In [ ]:
for plan in plans:
    if plan.delivery != an.DELIVERY_TEXT_CARD:
        continue

    parts = []
    for segment in an.segment_bulletin(bulletin, plan.char_cap):
        response = client.text.translate(
            input=segment,
            source_language_code="en-IN",
            target_language_code=plan.code,
            model=plan.translate_model,
            numerals_format="international",
        )
        parts.append(response.translated_text)

    translated = "".join(parts)
    report = an.audit_translation(facts, translated)
    card = an.render_text_card(plan, translated, report)

    card_path = OUTPUT_DIR / f"alert-{plan.code}-card.txt"
    card_path.write_text(card, encoding="utf-8")
    print(card)
    print()

## What this recipe does not tell you

A passing check means the numbers survived. It does not mean the translation is good. If km/h
comes back as miles per hour, or camp comes back as hospital, this check says nothing at all. It
reads numbers, and somebody who reads the language still has to read the alert.

It also cannot check a spoken-form rendering, and it does not pretend to: arm 3 above comes back
asking for a person, and that is the honest answer rather than a green tick.